# PA4
Group PA4 9, Melker Gustafsson, Pontus Gideflod, Ismail Sacic

## Task 1

In [1]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import Perceptron
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline

X1 = [{'city':'Gothenburg', 'month':'July'},
      {'city':'Gothenburg', 'month':'December'},
      {'city':'Paris', 'month':'July'},
      {'city':'Paris', 'month':'December'}]
Y1 = ['rain', 'rain', 'sun', 'rain']

X2 = [{'city':'Sydney', 'month':'July'},
      {'city':'Sydney', 'month':'December'},
      {'city':'Paris', 'month':'July'},
      {'city':'Paris', 'month':'December'}]
Y2 = ['rain', 'sun', 'sun', 'rain']

classifier1 = make_pipeline(DictVectorizer(), Perceptron(max_iter=10))
classifier1.fit(X1, Y1)
guesses1 = classifier1.predict(X1)
print(accuracy_score(Y1, guesses1))

classifier2 = make_pipeline(DictVectorizer(), Perceptron(max_iter=10))
#classifier2 = make_pipeline(DictVectorizer(), LinearSVC())
classifier2.fit(X2, Y2)
guesses2 = classifier2.predict(X2)
print(accuracy_score(Y2, guesses2))

1.0
0.5


### When we use the second set instead of the first, for some strange reason our classifier performs poorly! We can't improve it by switching to a LinearSVC. Do you have an idea what's going on? Why could the classifier "memorize" the training data in the first case, but not in the second case?

The DictVectorizer turns the dataset into points with either the result RAIN or SUN. the first dataset then becomes (0,1) = RAIN, (1,0) = SUN, (0,0) = RAIN and (1,1) = RAIN and if we plot this on a graph we can quickly see that a linear line can easily seperate the dataset into RAIN on one side and SUN on the other. On the second dataset however, we end up with (0,1) = SUN, (1,0) = SUN, (0,0) = RAIN and (1,1) = RAIN which means that the RAIN and SUN results are no longer seperable linearly (by a straight line). This causes the classifier to fail and in order to improve this you would need to change to a nonlinear model.

## Task 2

### The Perceptron package to install contains a Python program (doc_classification.py) that carries out an experiment in document classification. This is the same file that we used in one of the demonstrations for one of the lectures. The task here is to determine whether a product review is positive or negative. The program trains a classifier using our perceptron implementation, and then evaluates the classifier on a test set. Run the experiment code and make sure it works on your machine. Training should take at most a few seconds, and the accuracy should be about 0.80.

We had to make a small alteration to add a is_trained flag to the Perceptron class so that Sklearn could understand that the model was fit already when running predict. After that 
we ran the code and got Accuracy: 0.7919, Training time: 0.90 sec.

## Task 3

### Implement the Pegasos algorithm for training support vector classifiers by converting the pseudocode in Algorithm 1 in the clarification document (Figure 1 in the original Pegasos paper) into proper Python. Test your implementation by using your own classifier instead of the perceptron in doc_clasification.py. It's probably easiest if you start from the existing code, for instance by making a copy of the class Perceptron, and then just modify it to implement Algorithm 1.

In [2]:
import numpy as np
from sklearn.base import BaseEstimator
from sklearn.utils.validation import check_is_fitted
class LinearClassifier(BaseEstimator):
    """
    General class for binary linear classifiers. Implements the predict
    function, which is the same for all binary linear classifiers. There are
    also two utility functions.
    """

    def decision_function(self, X):
        """
        Computes the decision function for the inputs X. The inputs are assumed to be
        stored in a matrix, where each row contains the features for one
        instance.
        """
        return X.dot(self.w)

    def predict(self, X):
        """
        Predicts the outputs for the inputs X. The inputs are assumed to be
        stored in a matrix, where each row contains the features for one
        instance.
        """
        check_is_fitted(self, ['is_fitted_']) 
        
        scores = self.decision_function(X)

        out = np.select([scores >= 0.0, scores < 0.0],
                        [self.positive_class, self.negative_class],
                        default=self.negative_class) 
        return out

    def find_classes(self, Y):
        """
        Finds the set of output classes in the output part Y of the training set.
        If there are exactly two classes, one of them is associated to positive
        classifier scores, the other one to negative scores. If the number of
        classes is not 2, an error is raised.
        """
        classes = sorted(set(Y))
        if len(classes) != 2:
            raise Exception("this does not seem to be a 2-class problem")
        self.positive_class = classes[1]
        self.negative_class = classes[0]
        self.is_fitted_ = True

    def encode_outputs(self, Y):
        """
        A helper function that converts all outputs to +1 or -1.
        """
        return np.array([1 if y == self.positive_class else -1 for y in Y])

class Pegasos(LinearClassifier):
    """
    A straightforward implementation of the perceptron learning algorithm.
    """

    def __init__(self, n_iter=20, lambda_=None):
        """
        The constructor can optionally take a parameter n_iter specifying how
        many times we want to iterate through the training set.
        """
        self.n_iter = n_iter
        self.lambda_ = lambda_

    def fit(self, X, Y):
        """
        Train a linear classifier using the perceptron learning algorithm.
        """

        # First determine which output class will be associated with positive
        # and negative scores, respectively.
        self.find_classes(Y)

        # Convert all outputs to +1 (for the positive class) or -1 (negative).
        Ye = self.encode_outputs(Y)

        # If necessary, convert the sparse matrix returned by a vectorizer
        # into a normal NumPy matrix.
        if not isinstance(X, np.ndarray):
            X = X.toarray()
            
        lam = self.lambda_ if self.lambda_ is not None else 1.0 / X.shape[0]


        # Initialize the weight vector to all zeros.
        n_features, n_samples = X.shape[1], X.shape[0]
        self.w = np.zeros(n_features)
        
        t = 0
        # Pegasos algorithm:
        for epoch in range(self.n_iter):
            indices = np.random.permutation(n_samples)
            for i in indices:
                t += 1
                eta = 1.0 / (lam * t)
                
                # Compute the output score for this instance.
                x_i, y_i = X[i], Ye[i]
                score = x_i.dot(self.w)
                # If there was an error, update the weights.
                if y_i*score < 1:
                    self.w = (1 - eta * lam) * self.w + eta * y_i * x_i
                else:
                    self.w = (1 - eta * lam) * self.w

        return self


Above is our implementation of the Pegasos algorithm. We copied the Perceptron implementation given and altered the fit method to implement the Pegasos algorithm instead. 

In [3]:
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline
from sklearn.feature_selection import SelectKBest
from sklearn.model_selection import train_test_split

def read_data(corpus_file):
    X = []
    Y = []
    with open(corpus_file, encoding='utf-8') as f:
        for line in f:
            _, y, _, x = line.split(maxsplit=3)
            X.append(x.strip())
            Y.append(y)
    return X, Y


if __name__ == '__main__':
    X, Y = read_data('data/all_sentiment_shuffled.txt')

    Xtrain, Xtest, Ytrain, Ytest = train_test_split(X, Y, test_size=0.2,
                                                    random_state=0)
    N_p = 50
    lam_p = 1 /(N_p * N_p)

    Pegasos_pipeline = make_pipeline(
        TfidfVectorizer(),
        SelectKBest(k=1000),
        Normalizer(),
        Pegasos(n_iter=N_p, lambda_=lam_p),
    )

    t0 = time.time()
    Pegasos_pipeline.fit(Xtrain, Ytrain)
    t1 = time.time()
    Yguess_svc = Pegasos_pipeline.predict(Xtest)
    print('--- Task 3: Pegasos ---')
    print('Training time: {:.2f} sec.'.format(t1 - t0))
    print('Accuracy:      {:.4f}'.format(accuracy_score(Ytest, Yguess_svc)))

--- Task 3: Pegasos ---
Training time: 3.85 sec.
Accuracy:      0.8351


We used the above code to run and test our Pegasos implementation. The code is a slightly altered version of doc_classification. We tested out different parameter values for n_iter and lambda_ and the best accuracy we found was using n_iter = 50 and lambda_ = 1 / (n_iter * n_iter) which gave us an Accuracy of 0.8338. Higher values for n_iter seemed to reduce accuracy and lower did the same. Higher values for lambda reduced accuracy and while lower values of lambda could improve accuracy it could come at the risk of overfitting.

## Task 4

### As we saw in the lecture, the logistic regression classifier is based on an objective function that looks almost like the one used by the SVC. The difference is just in which loss function is used: the SVC uses the hinge loss and LR the log loss. Code a new training algorithm that uses the log loss instead of the hinge loss. Describe how well it works compared to your previous classifier.

In [4]:
class LogisticRegression(LinearClassifier):
    """
    Logistic Regression trained with Pegasos-style SGD (log loss).

    The update rule at step t is:
        eta  = 1 / (lambda * t)
        w   <- (1 - eta * lambda) * w  +  eta * y_i * sigma(-y_i * s_i) * x_i

    where s_i = w · x_i and sigma(-y_i*s_i) = 1 / (1 + exp(y_i * s_i)).

    Parameters
    ----------
    n_iter     : int   – number of epochs (passes through the training set)
    lambda_    : float – regularisation strength λ; defaults to 1/N
    """

    def __init__(self, n_iter=10, print_objective=True, lambda_=None):
        self.n_iter = n_iter
        self.lambda_ = lambda_
        self.print_objective = print_objective


    def fit(self, X, Y):
        self.find_classes(Y)
        Ye = self.encode_outputs(Y)

        # Convert sparse matrix (e.g. from TF-IDF) to dense array if needed.
        if not isinstance(X, np.ndarray):
            X = X.toarray()

        n_samples, n_features = X.shape
        lam = self.lambda_ if self.lambda_ is not None else 1.0 / n_samples
        self.w = np.zeros(n_features)
        t = 0  # global step counter

        for epoch in range(self.n_iter):
            # Shuffle each epoch for better convergence.
            epoch_loss = 0.0  # for tracking the total loss across epochs
            indices = np.random.permutation(n_samples)
            for i in indices:
                t += 1
                eta = 1.0 / (lam * t)
                x_i, y_i = X[i], Ye[i]
                margin = y_i * x_i.dot(self.w)

                # sigma(-margin) = 1 / (1 + exp(margin))
                
                
                sigmoid_neg = 1.0 / (1.0 + np.exp(margin))
                
                epoch_loss += np.log1p(np.exp(-margin))  # log-loss for this instance

                # L2 regularisation shrinkage + log-loss gradient step.
                self.w *= (1.0 - eta * lam)
                self.w += eta * y_i * sigmoid_neg * x_i

            # Objective: mean logistic loss + L2 regularizer
            if self.print_objective:
                mean_log_loss = epoch_loss / indices.size
                reg = 0.5 * lam * np.dot(self.w, self.w)
                obj = mean_log_loss + reg
                print(
                    f"Epoch {epoch + 1}/{self.n_iter} | "
                    f"objective≈{obj:.6f} (loss={mean_log_loss:.6f}, reg={reg:.6f})"
                )

        return self



Above you can see our implementation of Linear Regression using log loss instead of the SVC Hinge loss. 

In [5]:
if __name__ == '__main__':
    X, Y = read_data('data/all_sentiment_shuffled.txt')

    Xtrain, Xtest, Ytrain, Ytest = train_test_split(X, Y, test_size=0.2,
                                                    random_state=0)
    
    N_lr = 30
    lam_lr = 1 /  N_lr

    lr_pipeline = make_pipeline(
        TfidfVectorizer(),
        SelectKBest(k=1000),
        Normalizer(),
        LogisticRegression(n_iter=N_lr, lambda_=lam_lr),
    )

    t0 = time.time()
    lr_pipeline.fit(Xtrain, Ytrain)
    t1 = time.time()
    Yguess_lr = lr_pipeline.predict(Xtest)
    print()
    print('--- Task 4: Logistic Regression ---')
    print('Training time: {:.2f} sec.'.format(t1 - t0))
    print('Accuracy:      {:.4f}'.format(accuracy_score(Ytest, Yguess_lr)))



Epoch 1/30 | objective≈0.682645 (loss=0.671796, reg=0.010850)
Epoch 2/30 | objective≈0.681950 (loss=0.671110, reg=0.010840)
Epoch 3/30 | objective≈0.681819 (loss=0.670993, reg=0.010826)
Epoch 4/30 | objective≈0.681690 (loss=0.670862, reg=0.010827)
Epoch 5/30 | objective≈0.681691 (loss=0.670869, reg=0.010822)
Epoch 6/30 | objective≈0.681698 (loss=0.670876, reg=0.010823)
Epoch 7/30 | objective≈0.681677 (loss=0.670855, reg=0.010822)
Epoch 8/30 | objective≈0.681680 (loss=0.670858, reg=0.010822)
Epoch 9/30 | objective≈0.681673 (loss=0.670851, reg=0.010822)
Epoch 10/30 | objective≈0.681683 (loss=0.670861, reg=0.010822)
Epoch 11/30 | objective≈0.681673 (loss=0.670851, reg=0.010821)
Epoch 12/30 | objective≈0.681666 (loss=0.670845, reg=0.010821)
Epoch 13/30 | objective≈0.681661 (loss=0.670841, reg=0.010821)
Epoch 14/30 | objective≈0.681663 (loss=0.670843, reg=0.010820)
Epoch 15/30 | objective≈0.681665 (loss=0.670845, reg=0.010820)
Epoch 16/30 | objective≈0.681660 (loss=0.670840, reg=0.010820)
E

Above you can see the code that we used to run the Logistic Regression model. As you can see we used 30 iterations instead of 50 on this model because it yielded better accuracy. When we looked at the loss of each epoch when using 50 iterations we saw that the loss plateaued around 30 iterations so makes sence as to why that would be an optimal value for n_iter. The result we got was Accuracy: 0.0.7604 which is worse than our original Pegasos model.

## Bonus task 1.

### (a) Faster linear algebra operations The bottlenecks in the code are the linear algebra operations: computing the dot product, scaling the weight vector, and adding the feature vector to the weight vector. Try to speed up your code by using BLAS functions, which are available in scipy.linalg.blas. (For general information about BLAS, see the Wikipedia entry.) These functions are more efficient than normal NumPy linear algebra operations, because they avoid a number of safety checks carried out by NumPy. (The lack of these checks may cause Python to crash if you use the BLAS functions incorrectly.) 

In [6]:
import scipy.linalg.blas as blas

class Pegasos_opt(LinearClassifier):
    """
    A straightforward implementation of the perceptron learning algorithm.
    """

    def __init__(self, n_iter=20, lambda_=None):
        """
        The constructor can optionally take a parameter n_iter specifying how
        many times we want to iterate through the training set.
        """
        self.n_iter = n_iter
        self.lambda_ = lambda_

    def fit(self, X, Y):
        """
        Train a linear classifier using the perceptron learning algorithm.
        """

        # First determine which output class will be associated with positive
        # and negative scores, respectively.
        self.find_classes(Y)

        # Convert all outputs to +1 (for the positive class) or -1 (negative).
        Ye = self.encode_outputs(Y)

        # If necessary, convert the sparse matrix returned by a vectorizer
        # into a normal NumPy matrix.
        if not isinstance(X, np.ndarray):
            X = X.toarray()
            
        lam = self.lambda_ if self.lambda_ is not None else 1.0 / X.shape[0]


        # Initialize the weight vector to all zeros.
        n_samples = X.shape[0]
        self.w = np.zeros(X.shape[1])
        
        t = 0
        # X = np.ascontiguousarray(X, dtype=np.float64)
        # self.w = np.ascontiguousarray(self.w, dtype=np.float64)
        # Pegasos algorithm:
        for epoch in range(self.n_iter):
            indices = np.random.permutation(n_samples)
            for i in indices:
                t += 1
                eta = 1.0 / (lam * t)
                
                # Compute the output score for this instance.
                x_i, y_i = X[i], Ye[i]
                
                
                #score = blas.ddot(self.w, x_i) #Slower for some reason?
                score = x_i.dot(self.w)
                
                factor1 = (1 - eta * lam)
                factor2 = eta * y_i
                
                blas.dscal(factor1, self.w)
                # If there was an error, update the weights.
                if y_i*score < 1:
                    #self.w = factor1 * self.w + factor2 * x_i
                    blas.daxpy(x_i, self.w, a=factor2)

        return self

Above is our optimization of the Pegasos model using the BLAS library. We used all functions that were recommended except the ddot() function because in our testing it was significantly slower than np.dot().

In [7]:
if __name__ == '__main__':
    X, Y = read_data('data/all_sentiment_shuffled.txt')

    Xtrain, Xtest, Ytrain, Ytest = train_test_split(X, Y, test_size=0.2,
                                                    random_state=0)
    N_p = 50
    lam_p = 1 /(N_p * N_p)

    Pegasos_pipeline = make_pipeline(
        TfidfVectorizer(),
        SelectKBest(k=1000),
        Normalizer(),
        Pegasos(n_iter=N_p, lambda_=lam_p),
    )

    t0 = time.time()
    Pegasos_pipeline.fit(Xtrain, Ytrain)
    t1 = time.time()
    Yguess_svc = Pegasos_pipeline.predict(Xtest)
    print('--- Task 3: Pegasos ---')
    print('Training time: {:.2f} sec.'.format(t1 - t0))
    print('Accuracy:      {:.4f}'.format(accuracy_score(Ytest, Yguess_svc)))
    
    Pegasos_opt_pipeline = make_pipeline(
        TfidfVectorizer(),
        SelectKBest(k=1000),
        Normalizer(),
        Pegasos_opt(n_iter=N_p, lambda_=lam_p),
    )

    t0 = time.time()
    Pegasos_opt_pipeline.fit(Xtrain, Ytrain)
    t1 = time.time()
    Yguess_svc = Pegasos_opt_pipeline.predict(Xtest)
    print('--- Bonus Task: Optimized Pegasos ---')
    print('Training time: {:.2f} sec.'.format(t1 - t0))
    print('Accuracy:      {:.4f}'.format(accuracy_score(Ytest, Yguess_svc)))

--- Task 3: Pegasos ---
Training time: 3.41 sec.
Accuracy:      0.8330
--- Bonus Task: Optimized Pegasos ---
Training time: 2.84 sec.
Accuracy:      0.8342


Above you can see the code we used to compare the optimized versus non-optimized models. From the time stamps achieved non-optimized Training time: 3.20 sec vs optimized Training time: 2.82 sec we can see that there was around a 13.5% speedup between the two models.

### (b) Using sparse vectors. Remove the SelectKBest step from the pipeline and check the difference in training time. (This will change your accuracy a bit.) You may also add the option ngram_range=(1,2) to the TfidfVectorizer and see what happens. Our implementation is slow and wasteful in terms of memory if we use a large set of features. With sparse vectors, this can be improved. Follow the example of SparsePerceptron in the example code and implement sparse versions of the SVC or LR algorithms.

In [8]:
def add_sparse_to_dense(x, w, factor):
    """
    Adds a sparse vector x, scaled by some factor, to a dense vector.
    This can be seen as the equivalent of w += factor * x when x is a dense
    vector.
    """
    w[x.indices] += factor * x.data

def sparse_dense_dot(x, w):
    """
    Computes the dot product between a sparse vector x and a dense vector w.
    """
    return np.dot(w[x.indices], x.data)

class Pegasos_sparse_opt(LinearClassifier):
    """
    Sparse version of Pegasos_opt using x.indices / x.data helpers.
    Keeps the exact same update logic as Pegasos_opt.
    """

    def __init__(self, n_iter=20, lambda_=None):
        self.n_iter = n_iter
        self.lambda_ = lambda_

    def fit(self, X, Y):
        self.find_classes(Y)
        Ye = self.encode_outputs(Y)

        n_samples, n_features = X.shape
        lam = self.lambda_ if self.lambda_ is not None else 1.0 / n_samples

        # Dense weight vector, sparse input rows.
        self.w = np.zeros(n_features, dtype=np.float64)

        t = 0
        for epoch in range(self.n_iter):
            indices = np.random.permutation(n_samples)
            for i in indices:
                t += 1
                eta = 1.0 / (lam * t)

                x_i, y_i = X[i], Ye[i]
                score = sparse_dense_dot(x_i, self.w)

                factor1 = 1.0 - eta * lam
                factor2 = eta * y_i
                
                self.w *= factor1
                
                if y_i * score < 1.0:
                    # self.w = factor1 * self.w + factor2 * x_i
                    add_sparse_to_dense(x_i, self.w, factor2)

        return self

Above you can see the code for our sparse implementation of pegasos algorithm.

In [9]:
if __name__ == '__main__':
    X, Y = read_data('data/all_sentiment_shuffled.txt')

    Xtrain, Xtest, Ytrain, Ytest = train_test_split(X, Y, test_size=0.2,
                                                    random_state=0)
    N_p = 50
    lam_p = 1 /(N_p * N_p)

    Pegasos_sparse_opt_pipeline = make_pipeline(
        TfidfVectorizer(ngram_range=(1, 2)),
        Normalizer(),
        Pegasos_sparse_opt(n_iter=N_p, lambda_=lam_p),
    )

    t0 = time.time()
    Pegasos_sparse_opt_pipeline.fit(Xtrain, Ytrain)
    t1 = time.time()
    Yguess_svc = Pegasos_sparse_opt_pipeline.predict(Xtest)
    print('--- Bonus Task: Optimized Pegasos (Sparse) ---')
    print('Training time: {:.2f} sec.'.format(t1 - t0))
    print('Accuracy:      {:.4f}'.format(accuracy_score(Ytest, Yguess_svc)))

--- Bonus Task: Optimized Pegasos (Sparse) ---
Training time: 93.22 sec.
Accuracy:      0.8397


Above is the code used to run our sparse Pegasos model. This model achieved a Training time: 78.39 sec and an Accuracy: 0.8414. The training time is significantly larger than before but still decent considering we are now using every single feature and that the accuracy has increased.

### (c) Speeding up the scaling operation. At some places in the pseudocode, the whole weight vector is scaled by some factor. This can be a slow operation if our training set consists of high-dimensional, sparse vectors. Read section 4 in the clarification document, or section 2.4 in the original Pegasos paper, about a trick for speeding up the scaling operation. Modify your implementation so that it uses this trick and check if you can improve the training time.

In [10]:
class Pegasos_sparse_scale_opt(LinearClassifier):
    """
    Sparse version of Pegasos_opt using x.indices / x.data helpers.
    Keeps the exact same update logic as Pegasos_opt.
    """

    def __init__(self, n_iter=20, lambda_=None):
        self.n_iter = n_iter
        self.lambda_ = lambda_

    def fit(self, X, Y):
        self.find_classes(Y)
        Ye = self.encode_outputs(Y)

        n_samples, n_features = X.shape
        lam = self.lambda_ if self.lambda_ is not None else 1.0 / n_samples

        # Dense weight vector, sparse input rows.
        self.w = np.zeros(n_features, dtype=np.float64)
        
        scaler = 1.0

        t = 0
        for epoch in range(self.n_iter):
            indices = np.random.permutation(n_samples)
            for i in indices:
                t += 1
                eta = 1.0 / (lam * t)

                x_i, y_i = X[i], Ye[i]
                score = scaler * sparse_dense_dot(x_i, self.w)

                factor1 = 1.0 - eta * lam
                factor2 = (eta * y_i) / scaler
                
                # To prevent the weights from becoming too small, we can rescale them back up when the scaling factor drops below a certain threshold.
                if scaler * factor1 < 1e-8:
                    self.w *= scaler * factor1
                    scaler = 1.0
                else:
                    scaler *= factor1
                
                if y_i * score < 1.0:
                    # self.w = factor1 * self.w + factor2 * x_i
                    add_sparse_to_dense(x_i, self.w, factor2)

        self.w *= scaler
        return self

Above is our code for the scaling optimization of the sparse Pegasos algorithm model. 

In [11]:
if __name__ == '__main__':
    X, Y = read_data('data/all_sentiment_shuffled.txt')

    Xtrain, Xtest, Ytrain, Ytest = train_test_split(X, Y, test_size=0.2,
                                                    random_state=0)
    N_p = 50
    lam_p = 1 /(N_p * N_p)
    
    Pegasos_sparse_scale_opt_pipeline = make_pipeline(
        TfidfVectorizer(ngram_range=(1, 2)),
        Normalizer(),
        Pegasos_sparse_scale_opt(n_iter=N_p, lambda_=lam_p),
    )

    t0 = time.time()
    Pegasos_sparse_scale_opt_pipeline.fit(Xtrain, Ytrain)
    t1 = time.time()
    Yguess_svc = Pegasos_sparse_scale_opt_pipeline.predict(Xtest)
    print('--- Bonus Task: Optimized Pegasos (Sparse Scale) ---')
    print('Training time: {:.2f} sec.'.format(t1 - t0))
    print('Accuracy:      {:.4f}'.format(accuracy_score(Ytest, Yguess_svc)))

--- Bonus Task: Optimized Pegasos (Sparse Scale) ---
Training time: 26.09 sec.
Accuracy:      0.8410


Above is our code to run the optimized scaling version of the sparse Pegasos algorithm. It achieved a Training time: 25.63 sec and an Accuracy: 0.8410 which is a 305.9% speedup from the sparse version without the scaling optimization. This does however lower the accuracy from 0.8414 to 0.8410 but it's such a small amount that it doesn't have any significant impact.